In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!uv pip install --python /content/poseenv/bin/python \
    mmpose==1.3.2 --no-deps
# YOLOX-Nano runs through onnxruntime (falls back to CPU automatically if the GPU build can't load)
!uv pip install --python /content/poseenv/bin/python onnxruntime-gpu

In [ ]:
# ---- download the models once into Drive ----
from pathlib import Path
import urllib.request

DATA_ROOT = Path(r"/content/drive/MyDrive/hackathon/Data")
MMPOSE_ROOT = DATA_ROOT / "mmpose"
CKPT_DIR = MMPOSE_ROOT / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

YOLOX_ONNX = CKPT_DIR / "yolox_nano.onnx"
RTMPOSE_CKPT = CKPT_DIR / "rtmpose-m_ap10k.pth"
RTMPOSE_CFG = MMPOSE_ROOT / "configs/animal_2d_keypoint/rtmpose/ap10k/rtmpose-m_8xb64-210e_ap10k-256x256.py"

downloads = {
    YOLOX_ONNX: "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.onnx",
    RTMPOSE_CKPT: "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/"
                  "rtmpose-m_simcc-ap10k_pt-aic-coco_210e-256x256-7a041aa1_20230206.pth",
    RTMPOSE_CFG: "https://raw.githubusercontent.com/open-mmlab/mmpose/v1.3.2/configs/"
                 "animal_2d_keypoint/rtmpose/ap10k/rtmpose-m_8xb64-210e_ap10k-256x256.py",
}
for path, url in downloads.items():
    if path.is_file():
        print("exists:", path.name)
        continue
    path.parent.mkdir(parents=True, exist_ok=True)
    print("downloading", path.name, "...")
    urllib.request.urlretrieve(url, path)

In [ ]:
import torch
import cv2
import os
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image, ImageDraw
from tqdm import tqdm
import onnxruntime as ort
from mmpose.apis import init_model, inference_topdown

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: CUDA not available (is torch a +cpu build?)")

In [ ]:
import sys, types

stub = types.ModuleType("mmcv.ops")
stub.__path__ = []

def _missing(name):
    if name.startswith("__"):
        raise AttributeError(name)
    class _Unavailable:
        def __init__(self, *a, **k):
            raise RuntimeError(f"mmcv.ops.{name} is unavailable (mmcv has no compiled ops)")
    _Unavailable.__name__ = name
    return _Unavailable

stub.__getattr__ = _missing
sys.modules["mmcv.ops"] = stub

In [ ]:
def _isArrayLike(obj):

    return not type(obj) == str and hasattr(obj, '__iter__') and hasattr(obj, '__len__')

In [ ]:

class YOLOXNano:
    """
    Official Megvii YOLOX-Nano (COCO, 416x416) via onnxruntime.

    Returns, for every image, a DataFrame with the same columns YOLOv5's
    results.pandas().xyxy gave: xmin, ymin, xmax, ymax, confidence, class, name
    """

    STRIDES = (8, 16, 32)

    def __init__(self, onnx_path, conf=0.01, iou=0.45, classes=(19,),
                 input_size=(416, 416), device="cuda:0"):
        providers = ["CPUExecutionProvider"]
        if str(device).startswith("cuda") and "CUDAExecutionProvider" in ort.get_available_providers():
            providers.insert(0, "CUDAExecutionProvider")
        self.session = ort.InferenceSession(str(onnx_path), providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        self.input_h, self.input_w = input_size
        self.conf = conf
        self.iou = iou
        self.classes = list(classes) if classes is not None else None

        # grid / stride table used to decode the raw head outputs
        grids, strides = [], []
        for s in self.STRIDES:
            hs, ws = self.input_h // s, self.input_w // s
            xv, yv = np.meshgrid(np.arange(ws), np.arange(hs))
            grids.append(np.stack((xv, yv), 2).reshape(-1, 2))
            strides.append(np.full((hs * ws, 1), s))
        self.grids = np.concatenate(grids, 0).astype(np.float32)
        self.strides = np.concatenate(strides, 0).astype(np.float32)

    def _preprocess(self, img):
        """YOLOX letterbox: resize keeping aspect, paste top-left on a 114-grey canvas. BGR, 0-255."""
        h, w = img.shape[:2]
        r = min(self.input_h / h, self.input_w / w)
        resized = cv2.resize(img, (int(w * r), int(h * r)), interpolation=cv2.INTER_LINEAR)
        canvas = np.full((self.input_h, self.input_w, 3), 114, dtype=np.uint8)
        canvas[: resized.shape[0], : resized.shape[1]] = resized
        blob = canvas.transpose(2, 0, 1)[None].astype(np.float32)
        return np.ascontiguousarray(blob), r

    def _detect(self, img):
        blob, r = self._preprocess(img)
        out = self.session.run(None, {self.input_name: blob})[0][0]   # (3549, 85)

        # decode: xy offsets on the grid, exp() for w/h
        xy = (out[:, :2] + self.grids) * self.strides
        wh = np.exp(out[:, 2:4]) * self.strides
        obj = out[:, 4:5]
        cls_scores = out[:, 5:] * obj

        cls_ids = cls_scores.argmax(1)
        scores = cls_scores[np.arange(len(cls_ids)), cls_ids]

        keep = scores >= self.conf
        if self.classes is not None:
            keep &= np.isin(cls_ids, self.classes)
        if not keep.any():
            return pd.DataFrame(columns=["xmin", "ymin", "xmax", "ymax", "confidence", "class", "name"])

        xy, wh, scores, cls_ids = xy[keep], wh[keep], scores[keep], cls_ids[keep]
        boxes = np.concatenate([xy - wh / 2, xy + wh / 2], 1) / r   # back to original image pixels

        H, W = img.shape[:2]
        boxes[:, [0, 2]] = boxes[:, [0, 2]].clip(0, W)
        boxes[:, [1, 3]] = boxes[:, [1, 3]].clip(0, H)

        # class-aware NMS
        idx = cv2.dnn.NMSBoxesBatched(
            np.c_[boxes[:, :2], boxes[:, 2:] - boxes[:, :2]].tolist(),   # x, y, w, h
            scores.tolist(), cls_ids.tolist(), self.conf, self.iou
        )
        idx = np.array(idx).reshape(-1)

        return pd.DataFrame({
            "xmin": boxes[idx, 0], "ymin": boxes[idx, 1],
            "xmax": boxes[idx, 2], "ymax": boxes[idx, 3],
            "confidence": scores[idx], "class": cls_ids[idx],
            "name": ["cow" if c == 19 else str(c) for c in cls_ids[idx]],
        })

    def __call__(self, paths):
        out = []
        for p in paths:
            img = cv2.imread(str(p))
            if img is None:
                raise IOError(f"Could not read {p}")
            out.append(self._detect(img))
        return out


In [ ]:
# AP-10K: 17 keypoints
AP10K_KEYPOINTS = [
    "L_Eye", "R_Eye", "Nose", "Neck", "Root_of_tail",
    "L_Shoulder", "L_Elbow", "L_F_Paw",
    "R_Shoulder", "R_Elbow", "R_F_Paw",
    "L_Hip", "L_Knee", "L_B_Paw",
    "R_Hip", "R_Knee", "R_B_Paw",
]
# used only if the checkpoint/config doesn't carry skeleton metadata
AP10K_SKELETON = [
    (0, 1), (0, 2), (1, 2), (2, 3), (3, 4),
    (3, 5), (5, 6), (6, 7), (3, 8), (8, 9), (9, 10),
    (4, 11), (11, 12), (12, 13), (4, 14), (14, 15), (15, 16),
]


In [ ]:
class Databuilder:
    """
    YOLOX-Nano cow detection -> dynamic crop around the cow -> resize to a COMMON size
    -> RTMPose-m (AP-10K, 17 keypoints).

    width, height : size every saved crop is resized to (also sets the crop aspect ratio, so a
                    square output gives a square crop and nothing is stretched).
    pad           : margin added on each side of the YOLO box, as a fraction of the box size.
    size_mode     : "frame" -> crop size follows the cow in every frame (cow always fills the crop).
                    "video" -> one crop size per video (largest cow in the video + pad); the crop
                               still follows the cow's position but its scale is constant, which keeps
                               keypoint units consistent across frames (better for temporal models).
    overwrite     : re-run a video even if its crop annotation CSV already exists.
    pose_bbox     : "det"  -> RTMPose gets the YOLOX box mapped into the crop (tighter, recommended).
                    "full" -> RTMPose gets the whole crop as the box (old behaviour).
    kpt_thr       : score threshold used only for drawing the visualisation.
    """

    def __init__(
        self, img_file, save_root, width, height,
        load_model=True, pad=0.15, size_mode="frame", overwrite=False,
        device=None, yolo_batch=16, det_conf=0.01, det_classes=(19,),
        pose_bbox="det", kpt_thr=0.3
    ):
        assert size_mode in ("frame", "video"), "size_mode must be 'frame' or 'video'"
        assert pose_bbox in ("det", "full"), "pose_bbox must be 'det' or 'full'"
        self.img = img_file
        self.save_root = Path(save_root)
        self.crop_w = int(width)    # output size
        self.crop_h = int(height)
        self.pad = float(pad)
        self.size_mode = size_mode
        self.overwrite = overwrite
        self.device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        self.yolo_batch = int(yolo_batch)
        self.det_conf = float(det_conf)
        self.det_classes = det_classes   # COCO ids; 19 = cow
        self.pose_bbox = pose_bbox
        self.kpt_thr = float(kpt_thr)

        self.set_save_root(save_root)

        self.model = None
        self.pose_model = None

        if load_model:
            self.load_yolo()

    def set_save_root(self, save_root):
        """Point all output folders at a new root (lets one loaded model be reused across videos)."""
        self.save_root = Path(save_root)
        self.crop_save = self.save_root / "annotation" / "crop"
        self.keypoints_save = self.save_root / "annotation" / "keypoints"
        self.img_save = self.save_root / "crop_img"
        self.keypoints_vis = self.save_root / "keypoint_img"

        for folder in (
            self.crop_save,
            self.keypoints_save,
            self.img_save,
            self.keypoints_vis
        ):
            folder.mkdir(parents=True, exist_ok=True)

    def load_yolo(self):
        print("Loading YOLOX-Nano...")

        onnx_path = Path(r"/content/drive/MyDrive/hackathon/Data/mmpose/checkpoints/yolox_nano.onnx")
        if not onnx_path.is_file():
            raise FileNotFoundError(f"Missing: {onnx_path}")

        self.model = YOLOXNano(
            onnx_path,
            conf=self.det_conf,
            iou=0.45,
            classes=self.det_classes,
            device=self.device
        )
        print("  onnxruntime providers:", self.model.session.get_providers())

    def load_pose(self):
        if self.pose_model is not None:
            return

        root = Path(r"/content/drive/MyDrive/hackathon/Data/mmpose")

        config = root / (
            "configs/animal_2d_keypoint/rtmpose/ap10k/"
            "rtmpose-m_8xb64-210e_ap10k-256x256.py"
        )

        checkpoint = root / "checkpoints" / "rtmpose-m_ap10k.pth"

        for path in (config, checkpoint):
            if not path.is_file():
                raise FileNotFoundError(f"Missing: {path}")

        print("Loading RTMPose-m (AP-10K)...")

        self.pose_model = init_model(
            str(config),
            str(checkpoint),
            device=self.device
        )

        n_kpt = self.pose_model.dataset_meta.get("num_keypoints", len(AP10K_KEYPOINTS))
        if n_kpt != 17:
            raise RuntimeError(f"Expected 17 AP-10K keypoints, model has {n_kpt}")

        names = self.pose_model.dataset_meta.get("keypoint_id2name")
        self.kpt_names = (
            [names[i] for i in range(n_kpt)] if names else AP10K_KEYPOINTS
        )
        self.skeleton = (
            self.pose_model.dataset_meta.get("skeleton_links") or AP10K_SKELETON
        )

    # ------------------------------------------------------------------
    # Crop geometry
    # ------------------------------------------------------------------
    def _crop_size(self, bw, bh):
        """(w, h) of a crop with the output aspect ratio that contains the padded YOLO box."""
        aspect = self.crop_w / self.crop_h
        need_w = bw * (1 + 2 * self.pad)
        need_h = bh * (1 + 2 * self.pad)
        cw = max(need_w, need_h * aspect)
        return cw, cw / aspect

    @staticmethod
    def _place(center, size, limit):
        """Top-left coordinate along one axis. Shifted to stay inside the image when the crop fits;
        if the crop is larger than the image it is centred and the overhang is padded black."""
        start = center - size / 2
        if size <= limit:
            start = np.clip(start, 0, limit - size)
        return int(round(start))

    def crop_images(self, img_dir, out_dir):
        if self.model is None:
            self.load_yolo()

        img_dir = Path(img_dir)
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        csv_path = self.crop_save / f"{img_dir.name}.csv"
        if csv_path.exists() and not self.overwrite:
            print(f"Crops already exist for {img_dir.name} (overwrite=False) - skipping")
            return

        img_files = sorted(
            p for p in img_dir.iterdir()
            if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        )

        # ---- pass 1: detect the cow in every frame (YOLOX-Nano) ----
        dets = []
        bs = max(1, self.yolo_batch)
        for s in tqdm(range(0, len(img_files), bs), desc="YOLOX detection"):
            batch = img_files[s:s + bs]
            det_list = self.model(batch)                    # one DataFrame per image

            for img_path, detections in zip(batch, det_list):
                if detections.empty:
                    print(f"No cow detected: {img_path.name}")
                    continue

                # Select highest-confidence cow
                bb = detections.loc[detections["confidence"].idxmax()]

                with Image.open(img_path) as im:
                    width, height = im.size

                bw = float(bb["xmax"] - bb["xmin"])
                bh = float(bb["ymax"] - bb["ymin"])
                cw, ch = self._crop_size(bw, bh)
                dets.append({
                    "path": img_path, "bb": bb, "W": width, "H": height,
                    "cx": float((bb["xmin"] + bb["xmax"]) / 2),
                    "cy": float((bb["ymin"] + bb["ymax"]) / 2),
                    "cw": cw, "ch": ch,
                })

        if not dets:
            return

        # one crop size for the whole video (position still follows the cow)
        if self.size_mode == "video":
            cw_v = max(d["cw"] for d in dets)
            for d in dets:
                d["cw"], d["ch"] = cw_v, cw_v * self.crop_h / self.crop_w

        # ---- pass 2: crop, resize to the common size, save ----
        annotations = []
        for d in tqdm(dets, desc="Cropping"):
            cw, ch = int(round(d["cw"])), int(round(d["ch"]))
            x0 = self._place(d["cx"], cw, d["W"])
            y0 = self._place(d["cy"], ch, d["H"])

            with Image.open(d["path"]) as im:
                im = im.convert("RGB")
                # PIL pads out-of-image areas with black, so the geometry stays exact
                cropped = im.crop((x0, y0, x0 + cw, y0 + ch))
                cropped = cropped.resize((self.crop_w, self.crop_h), Image.LANCZOS)
                cropped.save(out_dir / d["path"].name)

            bb = d["bb"]
            annotations.append({
                "image": d["path"].name,
                "xmin": float(bb["xmin"]),
                "ymin": float(bb["ymin"]),
                "xmax": float(bb["xmax"]),
                "ymax": float(bb["ymax"]),
                "confidence": float(bb["confidence"]),
                "xcrop": x0,          # crop top-left in the ORIGINAL frame
                "ycrop": y0,
                "crop_w": cw,         # crop size in the ORIGINAL frame (before resizing)
                "crop_h": ch,
            })

        df = pd.DataFrame(annotations)
        name = img_dir.name

        df.to_csv(csv_path, index=False)
        df.to_hdf(
            self.crop_save / f"{name}.h5",
            key="df_with_missing",
            mode="w"
        )

    # ------------------------------------------------------------------
    # Pose
    # ------------------------------------------------------------------
    def _crop_lookup(self, name):
        """image name -> crop geometry + detector box (original-frame pixels), if available."""
        p = self.crop_save / f"{name}.csv"
        if not p.is_file():
            return {}
        c = pd.read_csv(p)
        if "crop_w" not in c.columns:   # CSV from the old fixed-size version
            return {}
        return {
            r.image: dict(x0=r.xcrop, y0=r.ycrop, cw=r.crop_w, ch=r.crop_h,
                          box=(r.xmin, r.ymin, r.xmax, r.ymax))
            for r in c.itertuples()
        }

    def run_pose(self, crop_dir, pose_vis=True):
        self.load_pose()

        crop_dir = Path(crop_dir)
        vis_dir = self.keypoints_vis / crop_dir.name
        vis_dir.mkdir(parents=True, exist_ok=True)

        crop_geo = self._crop_lookup(crop_dir.name)

        images = sorted(
            p for p in crop_dir.iterdir()
            if p.suffix.lower() in (".jpg", ".jpeg", ".png")
        )

        records = []

        for img_path in tqdm(images, desc="Pose estimation"):
            with Image.open(img_path) as im:
                width, height = im.size

            geo = crop_geo.get(img_path.name)

            # box handed to the top-down model, in resized-crop pixels
            if self.pose_bbox == "det" and geo is not None:
                sx, sy = width / geo["cw"], height / geo["ch"]
                xmin, ymin, xmax, ymax = geo["box"]
                box = [
                    (xmin - geo["x0"]) * sx, (ymin - geo["y0"]) * sy,
                    (xmax - geo["x0"]) * sx, (ymax - geo["y0"]) * sy,
                ]
                box = np.clip(box, 0, [width, height, width, height])
            else:
                box = [0, 0, width, height]

            bbox = np.array([box], dtype=np.float32)

            results = inference_topdown(
                self.pose_model,
                str(img_path),
                bboxes=bbox,
                bbox_format="xyxy"
            )

            if not results:
                continue

            pred = results[0].pred_instances
            keypoints = pred.keypoints[0]
            scores = pred.keypoint_scores[0]

            # map crop coordinates back to the original frame
            for i, ((x, y), score) in enumerate(
                zip(keypoints, scores)
            ):
                x, y = float(x), float(y)
                if geo is not None:
                    x_full = geo["x0"] + x * geo["cw"] / width
                    y_full = geo["y0"] + y * geo["ch"] / height
                else:
                    x_full = y_full = float("nan")

                records.append({
                    "image": img_path.name,
                    "keypoint": i,
                    "name": self.kpt_names[i],
                    "x": x,               # crop coordinates (resized crop)
                    "y": y,
                    "score": float(score),
                    "x_full": x_full,     # original-frame coordinates
                    "y_full": y_full,
                })

            if pose_vis:
                image = Image.open(img_path).convert("RGB")
                draw = ImageDraw.Draw(image)

                # Draw skeleton connections (AP-10K)
                for a, b in self.skeleton:
                    if scores[a] < self.kpt_thr or scores[b] < self.kpt_thr:
                        continue

                    draw.line(
                        [
                            tuple(map(float, keypoints[a])),
                            tuple(map(float, keypoints[b]))
                        ],
                        fill="lime",
                        width=3
                    )

                # Draw keypoints
                for i, ((x, y), score) in enumerate(
                    zip(keypoints, scores)
                ):
                    if score < self.kpt_thr:
                        continue

                    x, y = float(x), float(y)
                    r = 5

                    draw.ellipse(
                        (x-r, y-r, x+r, y+r),
                        fill="red"
                    )

                    draw.text(
                        (x+7, y+7),
                        str(i),
                        fill="yellow"
                    )

                output_path = vis_dir / (
                    img_path.stem + "_pose.png"
                )
                image.save(output_path)

        if records:
            df = pd.DataFrame(records)
            name = crop_dir.name

            df.to_csv(
                self.keypoints_save / f"{name}.csv",
                index=False
            )
            df.to_hdf(
                self.keypoints_save / f"{name}.h5",
                key="keypoints",
                mode="w"
            )

            print(f"Saved keypoints for {name}")

    def pipeline(
        self, needcrop=True, pose=True, pose_vis=True
    ):
        image_dirs = (
            [self.img]
            if isinstance(self.img, (str, Path))
            else self.img
        )

        for img_dir in image_dirs:
            img_dir = Path(img_dir)

            if not img_dir.is_dir():
                raise NotADirectoryError(img_dir)

            crop_dir = self.img_save / img_dir.name

            if needcrop:
                self.crop_images(img_dir, crop_dir)

            if pose:
                if not crop_dir.is_dir():
                    print(f"No cropped images: {crop_dir}")
                    continue

                self.run_pose(crop_dir, pose_vis)

In [ ]:
import cv2
import os

def video_to_frames(video_path, output_folder="test_images", target_fps=15):
    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    original_fps = cap.get(cv2.CAP_PROP_FPS)

    if original_fps <= 0:
        raise ValueError("Could not determine video FPS")

    print(f"Original FPS: {original_fps}")
    print(f"Target FPS: {target_fps}")

    frame_number = 0
    saved_number = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        current_time = frame_number / original_fps
        expected_saved_frames = int(current_time * target_fps) + 1

        if saved_number < expected_saved_frames:
            frame_name = f"{saved_number:06d}.jpg"
            frame_path = os.path.join(output_folder, frame_name)

            cv2.imwrite(frame_path, frame)
            saved_number += 1

        frame_number += 1

    cap.release()

    print(f"Done! Saved {saved_number} frames at {target_fps} FPS.")

In [ ]:
pwd

In [ ]:
home = Path(r"/content/drive/MyDrive/hackathon/Data")

lame = Path(str(Path) + r"\Lame")
normal = Path(str(Path) + r"\Normal")

lame = home / "Lame"
normal = home / "Normal"

print(normal.is_dir())  # True
print(lame.is_dir())    # True

if True:
    for folder in [normal, lame]:
        for video in folder.glob("*.mp4"):

            output_folder = folder / video.stem
            output_folder.mkdir(parents=True, exist_ok=True)

            video_to_frames(
                video_path=str(video),
                output_folder=str(output_folder),
                target_fps=10
            )

In [ ]:
target_width = 360

if True:
    for category in [home / "Normal", home / "Lame"]:
        for folder in category.iterdir():

            if not folder.is_dir():
                continue

            for image_path in folder.glob("*.jpg"):
                img = cv2.imread(str(image_path))

                if img is None:
                    print(f"Could not read: {image_path}")
                    continue

                h, w = img.shape[:2]

                # Preserve aspect ratio
                new_height = round(h * target_width / w)

                # Use appropriate interpolation
                interpolation = (
                    cv2.INTER_AREA if w > target_width
                    else cv2.INTER_CUBIC
                )

                resized = cv2.resize(
                    img,
                    (target_width, new_height),
                    interpolation=interpolation
                )

                # Overwrite original frame
                cv2.imwrite(str(image_path), resized)

            print(f"Done: {folder}")

In [ ]:
import time
import shutil
import traceback
from pathlib import Path

# "data_ap10k" lives next to this notebook. Kept separate from the old AnimalPose (20-kp) output
# so the resume check below doesn't skip videos that were done with the old models.
project_dir = Path.cwd()
data_out = project_dir / "data_ap10k"
work_root = data_out / "_work"

for label in ["Lame", "Normal"]:
    (data_out / label).mkdir(parents=True, exist_ok=True)
work_root.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE, "| output:", data_out)

# load YOLOX-Nano once and reuse it for every video.
# 256x256 crops = RTMPose-m AP-10K input size.
builder = Databuilder(
    [], work_root, 256, 256, size_mode="video", device=DEVICE, yolo_batch=16,
    det_conf=0.01, det_classes=(19,),   # COCO 19 = cow
    pose_bbox="det", kpt_thr=0.3,
)
builder.load_pose()   # crash immediately if the pose model can't load

failed = []

for label, folder in [("Normal", normal), ("Lame", lame)]:
    video_dirs = sorted(f for f in folder.iterdir() if f.is_dir())

    for i, video_dir in enumerate(video_dirs, 1):
        name = video_dir.name
        target_csv = data_out / label / f"{name}.csv"

        # resume support
        if target_csv.exists():
            print(f"[{label} {i}/{len(video_dirs)}] skip (done): {name}")
            continue

        print(f"[{label} {i}/{len(video_dirs)}] processing: {name}")
        start = time.time()

        try:
            builder.img = [str(video_dir)]
            builder.set_save_root(work_root / label / name)
            builder.pipeline(needcrop=True, pose=True, pose_vis=True)
        except Exception:
            print(f"FAILED: {label}/{name}")
            traceback.print_exc()
            failed.append((label, name))
            continue

        csv_path = builder.keypoints_save / f"{name}.csv"
        if not csv_path.is_file():
            print(f"No keypoints CSV found for {label}/{name}")
            failed.append((label, name))
            continue

        shutil.copy2(csv_path, target_csv)
        print(f"  saved -> {target_csv}  ({time.time() - start:.0f}s)")

print(f"\nFinished. Failed: {len(failed)}")
for label, name in failed:
    print(f"  {label}/{name}")